# Rainfall Analysis

This notebook cleans the Machakos rainfall record, combines the county data into a NumPy array, and identifies the best planting windows.


In [1]:
import numpy as np
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],
]
machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ", "9", "12", "", "45", "182", "96"]
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
valid_values = []
for value in machakos_raw:
    try:
        valid_values.append(int(value.strip()))
    except ValueError:
        pass
mean_value = round(sum(valid_values) / len(valid_values))
cleaned = []
for value in machakos_raw:
    try:
        cleaned.append(int(value.strip()))
    except ValueError:
        cleaned.append(mean_value)
imputed_months = [month_names[i] for i, value in enumerate(machakos_raw) if value.strip() in {"", "N/A"}]
print("Machakos cleaned:", cleaned)
print("Imputed months:", imputed_months)


Machakos cleaned: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]
Imputed months: ['Apr', 'Sep']


In [2]:
rainfall = np.array(clean_rows + [cleaned], dtype=int)
print(rainfall)
print(f"Shape: {rainfall.shape}")
print(f"Dtype: {rainfall.dtype}")


[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
Dtype: int64


In [3]:
annual_totals = rainfall.sum(axis=1)
monthly_averages = rainfall.mean(axis=0)
wettest_idx = np.argmax(rainfall, axis=1)
wettest_months = [month_names[i] for i in wettest_idx]
print("Annual totals (mm):", annual_totals)
print("Monthly averages (mm):", monthly_averages)
print("Wettest month by county:", list(zip(counties, wettest_months)))


Annual totals (mm): [1118  930 1571 1261  992]
Monthly averages (mm): [47.  48.  91.  164.4 155.2  74.6  60.6  69.4  66.4  65.8 107.4  77.8]
Wettest month by county: [('Nairobi', 'Apr'), ('Nakuru', 'Apr'), ('Kisumu', 'Apr'), ('Mombasa', 'May'), ('Machakos', 'Nov')]


In [4]:
dry_months = (rainfall < 50).sum(axis=1)
window_sums = rainfall[:, :-1] + rainfall[:, 1:]
best_window_idx = np.argmax(window_sums, axis=1)
planting_windows = [f"{month_names[i]}-{month_names[i + 1]}" for i in best_window_idx]
print("Dry month counts:", dry_months)
print("Planting windows:", list(zip(counties, planting_windows)))


Dry month counts: [4 3 0 0 5]
Planting windows: [('Nairobi', 'Apr-May'), ('Nakuru', 'Apr-May'), ('Kisumu', 'Apr-May'), ('Mombasa', 'Apr-May'), ('Machakos', 'Oct-Nov')]


In [5]:
import csv
summary_rows = [
    {
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": wettest,
        "dry_months": int(dry_count),
        "planting_window": window,
    }
    for county, total, wettest, dry_count, window in zip(counties, annual_totals, wettest_months, dry_months, planting_windows)
]
with open("rainfall_summary.csv", "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    writer.writeheader()
    writer.writerows(summary_rows)
print("rainfall_summary.csv written successfully")


rainfall_summary.csv written successfully


## Planting Advisory

For most counties, the strongest planting periods are clustered in the late wet season, so farmers should focus on the two-month windows with the highest combined rainfall.
Nairobi, Nakuru and Kisumu all peak around April-May, while Mombasa is strongest in April-May and Machakos is strongest in October-November.
These windows reduce the risk of crop stress and improve the odds of successful germination when rainfall is most dependable.
The NGO should reinforce crop planning around these periods and encourage farmers to monitor short dry spells during the transition months.


## Data Quality Notes

The Machakos dataset contained handwritten issues and missing entries, so the invalid values were replaced with the rounded average of the valid readings.
This means the cleaned record is a practical estimate rather than a perfect measurement, and local field checks should be used to confirm early-season rainfall.
The rainfall totals and planting advice are therefore best treated as operational guidance, not exact forecasting outputs.
